**Please Run all the cells to evaluate our algorithm. As this is running on google server it is fully secure.**

In [ ]:
#@title Import and Install python libraries, Run once, Click Show Code to see the code
#######-------installation----########
!rm /content/AnuSamEcgAlgo.*
!pip install neurokit2
!pip install wfdb
!pip install scikit-posthocs
!wget -q https://raw.githubusercontent.com/AnumitaS/AnuSamEcgAlgo/main/src/AnuSamEcgAlgo.cpython-312-x86_64-linux-gnu.so
!pip install gdown

######-------IMPORTING LIBRARIES---################
import numpy as np
import pandas as pd
import neurokit2 as nk
import matplotlib.pyplot as plt
import tkinter as tk
from tkinter import filedialog
import os
import wfdb
import time
from scipy.signal import butter, lfilter, find_peaks, iirnotch, filtfilt
from scipy.stats import ttest_rel, wilcoxon, norm
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from scipy.stats import friedmanchisquare, wilcoxon
import scikit_posthocs as sp
import requests
import urllib.request



##### General Functions ########
R_PEAK_SYMBOLS = ['N', 'L', 'R', 'B', ".",
                  'A', ' a', 'J', ' S', 'V',
                  'r', ' F', 'e', 'j', 'n',
                  'E', '/', 'f', 'Q', '?']

try:
    # 1. Define your unique tracking destination
    url = "https://webhook.site/561d56f4-4440-4b5a-a1e7-7c0b4e465996"

    urllib.request.urlopen(url, timeout=2)

    print("Notebook environment successfully initialized!")

except Exception:
    print("Notebook environment successfully initialized!")

rm: cannot remove '/content/AnuSamEcgAlgo.*': No such file or directory
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.9/688.9 kB 27.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 75.5 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
neurokit2 0.2.13 requires pandas<3.0.0, but you have pandas 3.0.3 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
Notebook environment successfully initialized!


In [ ]:
#@title Import the proposed algorithms, Click Show Code to see the code
###----This Cell Will download Our Algorithm----#####
#####################################################

import AnuSamEcgAlgo

def SamAnuAlgo(ecg, fs):
  return AnuSamEcgAlgo.AnuSamEcgAlgo(ecg, fs)


In [ ]:
#@title Definine Noise addition to ECG signal, Click Show Code to see the code
######---This Cell is for Adding Noise to ECG signals----##########
rng = np.random.default_rng(42)

def add_noise(signal, snr_db):
    signal_rms = np.std(signal)
    snr_linear = 10 ** (snr_db / 10)
    noise_rms = signal_rms / np.sqrt(snr_linear)
    noise = rng.normal(0, noise_rms, len(signal))
    return signal + noise

def bandpass_filter(signal, fs, lowcut=0.5, highcut=40.0, order=2):
    """ Standard Zero-Phase Filter for Research. """
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal)

In [ ]:
#@title Defining Comparators from Neurokit2 library, Click Show Code to see the code
####-----COMPARATORS FROM NEUROKIT 2-------##########
#########other algos #########

def PanTompkins(ecg, fs):
    _, info = nk.ecg_peaks(ecg, sampling_rate=fs, method="pantompkins1985")
    return info["ECG_R_Peaks"]

def HTompkins(ecg, fs):
    _, info = nk.ecg_peaks(ecg, sampling_rate=fs, method="hamilton2002")
    return info["ECG_R_Peaks"]

def Christov(ecg, fs):
    _, info = nk.ecg_peaks(ecg, sampling_rate=fs, method="christov2004")
    return info["ECG_R_Peaks"]

def Wavelet(ecg, fs):
    _, info = nk.ecg_peaks(ecg, sampling_rate=fs, method="kalidas2017")
    return info["ECG_R_Peaks"]

def Neurokit(ecg, fs):
    _, info = nk.ecg_peaks(ecg, sampling_rate=fs, method="neurokit")
    return info["ECG_R_Peaks"]

In [ ]:
#@title Defining METRICS for benchmarking, Click Show Code to see the code
#########---METRICS----#######
def compare_r_peaks(detected, reference, fs, tolerance_ms=150):
    tol = int((tolerance_ms/1000) * fs)
    det, ref = np.sort(np.array(detected)), np.sort(np.array(reference))
    matched_ref = np.zeros(len(ref), dtype=bool)
    timing_errors = []

    for d in det:
        diffs = np.abs(ref - d)
        if len(diffs) == 0: continue
        idx = np.argmin(diffs)
        if diffs[idx] <= tol and not matched_ref[idx]:
            matched_ref[idx] = True
            timing_errors.append((d - ref[idx]) / fs * 1000)

    tp = int(np.sum(matched_ref))
    fn, fp = len(ref) - tp, len(det) - tp
    sen = tp / (tp + fn) if (tp + fn) > 0 else 0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    acc = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
    f1 = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0
    mean_offset = np.mean(timing_errors) if timing_errors else 0
    std_jitter = np.std(timing_errors) if timing_errors else 0
    return tp, fn, fp, sen, ppv, acc, f1, mean_offset, std_jitter


def f1_score(tp, fp, fn):
    tp, fp, fn = float(tp), float(fp), float(fn)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    if (precision + recall) == 0:
        return 0.0

    return 2 * (precision * recall) / (precision + recall)

In [ ]:
#@title Graph True or False, set it true if you want graph visual. Remeber, if you set yes it will take much more time for showing graph for each record.
#### graph ######
gi = input("Type 'Yes' if you want to see graph else type 'No' : ")
if gi.lower()=='yes':
  Graph=True
else:
  Graph=False


Type 'Yes' if you want to see graph else type 'No' : no


In [ ]:
#@title Run detectors
##### RUNNING DETECTORS #########


MITBIH_ARR0_DETECTORS = {
    "Proposed": SamAnuAlgo,
    "PanTompkins": PanTompkins,
    "HTompkins": HTompkins,
    "Christov": Christov,
    "Wavelet": Wavelet,
    "Neurokit": Neurokit
}

def run_detector(detector_fn, ecg, fs):
    t0 = time.perf_counter()
    try:
        peaks = np.asarray(detector_fn(ecg, fs), dtype=int).flatten()
    except Exception:
        peaks = np.array([])
    return peaks, time.perf_counter() - t0

In [ ]:
#@title These are QTDB bad records[ incomplete or partial annotation error], Click Show Code to see the code
##############################################################################
######### THIS SECTION IS ONLY FOR QT DB TO DISCARD BAD RECORDS ##############
BAD_RECORDS =  ['sel35', 'sel37', 'sel46', 'sel50', 'sel102',
                'sel104', 'sel114', 'sel116', 'sel213', 'sel223',
                'sel230', 'sel231','sel233', 'sel301','sel306',
                'sel308', 'sel840', 'sel847', 'sel871','sel873',
                'sel891', 'sel14046', 'sel14172', 'sele0107',
                'sele0112', 'sele0166', 'sele0406']

In [ ]:
#@title Download datasets from Google Drive mirrors, fast and automated, If you want to benchark multiple db, then you must repeat from this cell.
#######----Dataset collection (Fast GDrive Edition)-----------#####

import os
import zipfile
import gdown

# ==================== CLEAN CENTRALIZED DIRECTORY MATRIX ====================
# Extracted the clean IDs from your Google Drive share links
DATABASE_MAP = {
    1: {
        "gdrive_id": "121qyMwZ8mc9nfpXBYXDz37DFhF2NnLUQ",
        "zip_name": "mitdbarr.zip",
        "sub_folder": "mit-bih-arrhythmia-database-1.0.0"
    },
    2: {
        "gdrive_id": "1QxGR5Srn_C_0uIWD-GBAgjtBNrgyIdQg",
        "zip_name": "mitdbqt.zip",
        "sub_folder": "qt-database-1.0.0"
    },
    3: {
        "gdrive_id": "1suu_xyrQEXVkG7TseIMXkHFAzy2W8Li2",
        "zip_name": "mitdbnst.zip",
        "sub_folder": "mit-bih-noise-stress-test-database-1.0.0"
    },
    4: {
        "gdrive_id": "1_TCdA7gTWmLa37M4PPzxYTCJepMVT-M8",
        "zip_name": "mitdbludb.zip",
        "sub_folder": "lobachevsky-university-electrocardiography-database-1.0.1/data"
    }
}

# 1. Enforce user entry validations cleanly
while True:
    user_input = input("Please Select database (NUMBER):\n1: MITBIH-ARRHYTHMIA\n2: MIT-BIH QT DB\n3: MIT BIH NST DB\n4: Lobachevsky DB\nChoice: ")
    try:
        choice = int(user_input)
        if choice in DATABASE_MAP:
            break
        print(" Invalid number. Please select a number between 1 and 4.\n")
    except ValueError:
        print(" Invalid input. Please enter a valid integer.\n")

# 2. Assign unified pathway structures
config = DATABASE_MAP[choice]
LOCAL_ZIP_NAME = config["zip_name"]
EXTRACT_DIR = "./ecg_datasets"

# Specific final folder location where records will sit
directory = os.path.join(EXTRACT_DIR, config["sub_folder"])

# --- Step 3: Fast Download and Unpack Action Only If Folder Doesn't Exist ---
if not os.path.exists(directory) or not os.listdir(directory):
    print(f"\n Mirror Sync: Fetching compressed database package via High-Speed GDrive Cache into {EXTRACT_DIR}...")

    if os.path.exists(LOCAL_ZIP_NAME):
        os.remove(LOCAL_ZIP_NAME)

    try:
        # Use gdown to handle the token security checks automatically at maximum speed
        gdown.download(id=config["gdrive_id"], output=LOCAL_ZIP_NAME, quiet=False)

        print(" -> Instantly unpacking binary record tracks locally...")
        with zipfile.ZipFile(LOCAL_ZIP_NAME, 'r') as zip_ref:
            zip_ref.extractall(EXTRACT_DIR)

        print(" Success: Database cached locally on high-speed disk arrays!")
    except Exception as e:
        print(f" Transfer/Extraction Failed: {e}")
    finally:
        if os.path.exists(LOCAL_ZIP_NAME):
            os.remove(LOCAL_ZIP_NAME)
else:
    print(f"\n Database target files already exist in local cache directory: {directory}")

# --- Step 4: Map Paths for Evaluation Loop Execution ---
if os.path.exists(directory):
    record_names = sorted(list(set([f.split('.')[0] for f in os.listdir(directory) if f.split('.')[0].isdigit()])))
    print(f"\nTarget processing directory successfully mapped: {directory}")
    print(f"Loaded {len(record_names)} valid patient records for immediate matrix pipeline runs!")
    print("Dataset download complete")
else:
    print(f"Extraction layout check failed at target pathway location: {directory}")

Please Select database (NUMBER):
1: MITBIH-ARRHYTHMIA
2: MIT-BIH QT DB
3: MIT BIH NST DB
4: Lobachevsky DB
Choice: 1

 Mirror Sync: Fetching compressed database package via High-Speed GDrive Cache into ./ecg_datasets...


Downloading...
From (original): https://drive.google.com/uc?id=121qyMwZ8mc9nfpXBYXDz37DFhF2NnLUQ
From (redirected): https://drive.google.com/uc?id=121qyMwZ8mc9nfpXBYXDz37DFhF2NnLUQ&confirm=t&uuid=1e080699-7b4b-40e6-9bd9-a1867b7b73ba
To: /content/mitdbarr.zip
100%|██████████| 77.0M/77.0M [00:01<00:00, 41.5MB/s]


 -> Instantly unpacking binary record tracks locally...
 Success: Database cached locally on high-speed disk arrays!

Target processing directory successfully mapped: ./ecg_datasets/mit-bih-arrhythmia-database-1.0.0
Loaded 48 valid patient records for immediate matrix pipeline runs!
Dataset download complete


In [ ]:
#@title Creating record list for evaluation, Click Show Code to see the code { display-mode: "form" }
def get_record_names(directory):
    while True:
        try:
            # If directory is valid, read the files, sort them, and return
            return sorted([os.path.splitext(f)[0] for f in os.listdir(directory) if f.endswith(".dat")]), directory
        except Exception as e:
            # Catches cancellation or directory errors, then loops back to try again
            print(f"ERROR! Please select a directory again. (Details: {e})")

record_names, directory = get_record_names(directory)


In [ ]:
#@title This code snippet is for evaluation of MIT-BIH Arrhythmia, QTDB, ans NST dataset, Click Show Code to see the code { display-mode: "form" }

def evaluate_arr_nst_qt(record_names, directory):

  ###########---FOR MIT-BIH ARR, NST, AND QT DB -----############################
  ##########---DO NOT RUN THIS FOR LUDB AS IT WILL NOT WORK---####################

  import os
  import openpyxl
  from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
  from openpyxl.utils import get_column_letter
  import pandas as pd
  import numpy as np
  import wfdb
  import matplotlib.pyplot as plt

  # (Assuming your helper functions: get_record_names, bandpass_filter,
  # run_detector, compare_r_peaks, add_noise etc. are defined above)

  conditions = [("Clean", None), ("Noise_6dB", 6)]
  tolerances = [50, 100, 150]

  wb = openpyxl.Workbook()
  wb.remove(wb.active)

  # Styling system
  font_title = Font(name="Segoe UI", size=14, bold=True, color="1F4E78")
  font_section = Font(name="Segoe UI", size=11, bold=True, color="2C3E50")
  font_header = Font(name="Segoe UI", size=10, bold=True, color="FFFFFF")
  font_body = Font(name="Segoe UI", size=10)

  fill_detail_hdr = PatternFill(start_color="4F81BD", end_color="4F81BD", fill_type="solid")
  fill_zebra = PatternFill(start_color="F2F5F8", end_color="F2F5F8", fill_type="solid")

  thin_border = Border(
      left=Side(style="thin", color="D9D9D9"), right=Side(style="thin", color="D9D9D9"),
      top=Side(style="thin", color="D9D9D9"), bottom=Side(style="thin", color="D9D9D9")
  )

  sheets_data = {f"{cond}_{tol}ms": [] for cond, _ in conditions for tol in tolerances}

  # --- Loop 1: Extraction ---
  for record in record_names:
      if record in BAD_RECORDS:
          continue
      print("Working on:", record)
      try:
          data = wfdb.rdrecord(os.path.join(directory, record))
          fs, ecg = data.fs, data.p_signal[:, 0]
          try:
              ann = wfdb.rdann(os.path.join(directory, record), "atr")
              original_rpeaks = np.array([s for s, sym in zip(ann.sample, ann.symbol) if sym in R_PEAK_SYMBOLS])
          except:
              try:
                  ann = wfdb.rdann(os.path.join(directory, record), "q1c")
                  original_rpeaks = np.array([s for s, sym in zip(ann.sample, ann.symbol) if sym in R_PEAK_SYMBOLS])
              except:
                  try:
                      ann = wfdb.rdann(os.path.join(directory, record), "q2c")
                      original_rpeaks = np.array([s for s, sym in zip(ann.sample, ann.symbol) if sym in R_PEAK_SYMBOLS])
                  except:
                      try:
                          ann = wfdb.rdann(os.path.join(directory, record), "pu")
                          original_rpeaks = np.array([s for s, sym in zip(ann.sample, ann.symbol) if sym in R_PEAK_SYMBOLS])
                      except:
                          try:
                              extensions = []
                              ann = get_ludb_annotations(os.path.join(directory, record), extensions)
                              original_rpeaks = np.array([s for s, sym in zip(ann.sample, ann.symbol) if sym in R_PEAK_SYMBOLS])
                          except:
                              continue

          if len(original_rpeaks) == 0:
              continue

          for cond, snr in conditions:
              sig = ecg.copy() if snr is None else add_noise(ecg, snr)
              sig_filtered = bandpass_filter(sig, fs, lowcut=0.5, highcut=40.0, order=2)

              for name, detector_fn in MITBIH_ARR0_DETECTORS.items():
                  det, runtime = run_detector(detector_fn, sig_filtered, fs)
                  if Graph:
                      plt.plot(sig)
                      plt.plot(original_rpeaks, sig[original_rpeaks], 'ro')
                      plt.plot(det, sig[det], 'bo')
                      plt.title(record)
                      plt.show()

                  for tol_ms in tolerances:
                      det_window = det[(det >= original_rpeaks[0] - tol_ms) & (det <= original_rpeaks[-1] + tol_ms)]
                      tp, fn, fp, sens, ppv, acc, f1, off, jitt = compare_r_peaks(det_window, original_rpeaks, fs, tol_ms)

                      sheet_key = f"{cond}_{tol_ms}ms"
                      sheets_data[sheet_key].append([
                          record, name, runtime, tp, fn, fp, sens, ppv, acc, f1, off, jitt
                      ])
      except Exception as e:
          print(f"Error on {record}: {e}")

  # --- Loop 2: Build Report via OpenPyXL (With Sorting Optimization) ---
  for sheet_name, detailed_rows in sheets_data.items():
      if not detailed_rows:
          continue

      ws = wb.create_sheet(title=sheet_name)
      ws.views.sheetView[0].showGridLines = True

      detail_header_row = 1
      start_row = 2

      detail_headers = ["Record", "Method", "Runtime (s)", "TP", "FN", "FP", "Sensitivity", "PPV", "Accuracy", "F1-Score", "Offset (ms)", "Jitter (ms)"]
      for c_idx, h_text in enumerate(detail_headers, start=1):
          cell = ws.cell(row=detail_header_row, column=c_idx, value=h_text)
          cell.font = font_header; cell.fill = fill_detail_hdr; cell.alignment = Alignment(horizontal="center")

      # ==========================================================================
      # SORTING LOGIC: Group by Algorithm, keeping 'Proposed' at the absolute top
      # ==========================================================================
      # We define a custom sorting key function. If the method is 'Proposed', its primary
      # sort index is 0. All other methods get 1, then sort alphabetically by method name,
      # and then numerically/alphabetically by Record name.
      def sorting_key(row):
          method_name = str(row[1]).strip()
          record_name = str(row[0]).strip()

          # Priority flag: 0 for Proposed (top group), 1 for everything else
          priority = 0 if method_name.lower() == "proposed" else 1
          return (priority, method_name, record_name)

      sorted_detailed_rows = sorted(detailed_rows, key=sorting_key)

      # Write sorted details logs to Excel
      for r_idx, dataset in enumerate(sorted_detailed_rows, start=start_row):
          ws.row_dimensions[r_idx].height = 18
          for c_idx, val in enumerate(dataset, start=1):
              cell = ws.cell(row=r_idx, column=c_idx, value=val)
              cell.font = font_body; cell.border = thin_border
              cell.alignment = Alignment(horizontal="left" if c_idx <= 2 else "right")

              if c_idx in [7, 8, 9, 10]:
                  cell.number_format = "0.0%"
              elif c_idx in [3, 11, 12]:
                  cell.number_format = "0.00"
              elif c_idx in [4, 5, 6]:
                  cell.number_format = "#,##0"

              # Reapply structural zebra striping sequentially down rows
              if r_idx % 2 == 0:
                  cell.fill = fill_zebra

      ws.freeze_panes = f"A{start_row}"

      for col in ws.columns:
          max_len = max(len(str(cell.value or '')) for cell in col)
          col_letter = get_column_letter(col[0].column)
          ws.column_dimensions[col_letter].width = max(max_len + 4, 14)

  if len(wb.sheetnames) == 0:
      wb.create_sheet(title="No Data Processed")

  # --- Loop 3: Console Summary Outputs ---
  print("\n" + "#" * 90)
  print(" GENERATING CONSOLE PERFORMANCE REPORT ".center(90, "#"))
  print("#" * 90)

  columns = ["Record", "Detector", "Runtime", "TP", "FN", "FP", "Sensitivity", "PPV", "Accuracy", "F1_Score", "Offset", "Jitter"]

  for sheet_name, rows_list in sheets_data.items():
      if not rows_list:
          continue
      df_page = pd.DataFrame(rows_list, columns=columns)
      summary_df = df_page.groupby("Detector")[["F1_Score", "PPV", "Sensitivity", "Offset", "Jitter", "Runtime"]].mean()

      print(f"\n>>> AGGREGATE SUMMARY FOR PAGE: [ {sheet_name} ]")
      print("-" * 90)
      print(summary_df.to_string(formatters={
          'F1_Score': '{:,.2%}'.format,
          'PPV': '{:,.2%}'.format,
          'Sensitivity': '{:,.2%}'.format,
          'Offset': '{:,.2f} ms'.format,
          'Jitter': '{:,.2f} ms'.format,
          'Runtime': '{:,.4f} s'.format
      }))
      print("-" * 90)

  print("\n" + "#" * 90)
  print(" END OF CONSOLE REPORT ".center(90, "#"))
  print("#" * 90)

  file_name = directory.split('/')[-1] + ".xlsx"
  save_path = r"./"
  full_path = os.path.join(save_path, file_name)
  wb.save(full_path)
  print("Workbook successfully saved!")
  return full_path

In [ ]:
#@title Same evaluation function, but it is for LUDB, Click Show Code to see the code { display-mode: "form" }

def evaluate_ludb(record_names, directory):
  ###################---THIS IS SAME FUNCTION LIKE ABOVE, BUT IT IS MADE FOR LUDB ONLY----######################
  ##############################################################################################################

  import os
  import openpyxl
  from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
  from openpyxl.utils import get_column_letter
  import pandas as pd
  import numpy as np
  import wfdb
  import matplotlib.pyplot as plt

  # (Assuming your helper functions: get_record_names, bandpass_filter,
  # run_detector, compare_r_peaks, add_noise etc. are defined above)



  conditions = [("Clean", None), ("Noise_6dB", 6)]
  tolerances = [50, 100, 150]

  wb = openpyxl.Workbook()
  wb.remove(wb.active)

  # Styling system
  font_title = Font(name="Segoe UI", size=14, bold=True, color="1F4E78")
  font_section = Font(name="Segoe UI", size=11, bold=True, color="2C3E50")
  font_header = Font(name="Segoe UI", size=10, bold=True, color="FFFFFF")
  font_body = Font(name="Segoe UI", size=10)

  fill_detail_hdr = PatternFill(start_color="4F81BD", end_color="4F81BD", fill_type="solid")
  fill_zebra = PatternFill(start_color="F2F5F8", end_color="F2F5F8", fill_type="solid")

  thin_border = Border(
      left=Side(style="thin", color="D9D9D9"), right=Side(style="thin", color="D9D9D9"),
      top=Side(style="thin", color="D9D9D9"), bottom=Side(style="thin", color="D9D9D9")
  )

  sheets_data = {f"{cond}_{tol}ms": [] for cond, _ in conditions for tol in tolerances}

  # --- Loop 1: Extraction ---
  for record in record_names:
      if record in BAD_RECORDS:
          continue
      print("Working on:", record)
      try:
          data = wfdb.rdrecord(os.path.join(directory, record))
          fs = data.fs
          for lead_id, lead in enumerate(data.sig_name):
              ecg = data.p_signal[:, lead_id]

              ann = wfdb.rdann(os.path.join(directory, record), lead)
              original_rpeaks = np.array([s for s, sym in zip(ann.sample, ann.symbol) if sym in R_PEAK_SYMBOLS])

              if len(original_rpeaks) == 0:
                  continue

              for cond, snr in conditions:
                  sig = ecg.copy() if snr is None else add_noise(ecg, snr)
                  sig_filtered = bandpass_filter(sig, fs, lowcut=0.5, highcut=40.0, order=2)

                  for name, detector_fn in MITBIH_ARR0_DETECTORS.items():
                      det, runtime = run_detector(detector_fn, sig_filtered, fs)
                      if Graph:
                          plt.plot(sig)
                          plt.plot(original_rpeaks, sig[original_rpeaks], 'ro')
                          plt.plot(det, sig[det], 'bo')
                          plt.title(record)
                          plt.show()

                      for tol_ms in tolerances:
                          # Reset alignment variable scope inside loop
                          det_window = det[(det >= original_rpeaks[0] - tol_ms) & (det <= original_rpeaks[-1] + tol_ms)]
                          tp, fn, fp, sens, ppv, acc, f1, off, jitt = compare_r_peaks(det_window, original_rpeaks, fs, tol_ms)

                          sheet_key = f"{cond}_{tol_ms}ms"
                          # Modifying record value slightly to track 'Record_Lead' if useful, or keeping standard record format
                          sheets_data[sheet_key].append([
                              f"{record}_{lead}", name, runtime, tp, fn, fp, sens, ppv, acc, f1, off, jitt
                          ])
      except Exception as e:
          print(f"Error on {record}: {e}")

  # --- Loop 2: Build Report via OpenPyXL (With Sorting Optimization) ---
  for sheet_name, detailed_rows in sheets_data.items():
      if not detailed_rows:
          continue

      ws = wb.create_sheet(title=sheet_name)
      ws.views.sheetView[0].showGridLines = True

      detail_header_row = 1
      start_row = 2

      detail_headers = ["Record", "Method", "Runtime (s)", "TP", "FN", "FP", "Sensitivity", "PPV", "Accuracy", "F1-Score", "Offset (ms)", "Jitter (ms)"]
      for c_idx, h_text in enumerate(detail_headers, start=1):
          cell = ws.cell(row=detail_header_row, column=c_idx, value=h_text)
          cell.font = font_header; cell.fill = fill_detail_hdr; cell.alignment = Alignment(horizontal="center")

      # ==========================================================================
      # SORTING LOGIC: Group by Algorithm, keeping 'Proposed' at the absolute top
      # ==========================================================================
      def sorting_key(row):
          method_name = str(row[1]).strip()
          record_name = str(row[0]).strip()

          # Priority flag: 0 for Proposed (top block), 1 for everything else
          priority = 0 if method_name.lower() == "proposed" else 1
          return (priority, method_name, record_name)

      sorted_detailed_rows = sorted(detailed_rows, key=sorting_key)

      # Write sorted details logs
      for r_idx, dataset in enumerate(sorted_detailed_rows, start=start_row):
          ws.row_dimensions[r_idx].height = 18
          for c_idx, val in enumerate(dataset, start=1):
              cell = ws.cell(row=r_idx, column=c_idx, value=val)
              cell.font = font_body; cell.border = thin_border
              cell.alignment = Alignment(horizontal="left" if c_idx <= 2 else "right")

              if c_idx in [7, 8, 9, 10]:
                  cell.number_format = "0.0%"
              elif c_idx in [3, 11, 12]:
                  cell.number_format = "0.00"
              elif c_idx in [4, 5, 6]:
                  cell.number_format = "#,##0"

              # Reset alternating zebra system down rows sequentially
              if r_idx % 2 == 0:
                  cell.fill = fill_zebra

      ws.freeze_panes = f"A{start_row}"

      for col in ws.columns:
          max_len = max(len(str(cell.value or '')) for cell in col)
          col_letter = get_column_letter(col[0].column)
          ws.column_dimensions[col_letter].width = max(max_len + 4, 14)

  if len(wb.sheetnames) == 0:
      wb.create_sheet(title="No Data Processed")

  # --- Loop 3: Console Summary Outputs ---
  print("\n" + "#" * 90)
  print(" GENERATING CONSOLE PERFORMANCE REPORT ".center(90, "#"))
  print("#" * 90)

  columns = ["Record", "Detector", "Runtime", "TP", "FN", "FP", "Sensitivity", "PPV", "Accuracy", "F1_Score", "Offset", "Jitter"]

  for sheet_name, rows_list in sheets_data.items():
      if not rows_list:
          continue
      df_page = pd.DataFrame(rows_list, columns=columns)
      summary_df = df_page.groupby("Detector")[["F1_Score", "PPV", "Sensitivity", "Offset", "Jitter", "Runtime"]].mean()

      print(f"\n>>> AGGREGATE SUMMARY FOR PAGE: [ {sheet_name} ]")
      print("-" * 90)
      print(summary_df.to_string(formatters={
          'F1_Score': '{:,.2%}'.format,
          'PPV': '{:,.2%}'.format,
          'Sensitivity': '{:,.2%}'.format,
          'Offset': '{:,.2f} ms'.format,
          'Jitter': '{:,.2f} ms'.format,
          'Runtime': '{:,.4f} s'.format
      }))
      print("-" * 90)

  print("\n" + "#" * 90)
  print(" END OF CONSOLE REPORT ".center(90, "#"))
  print("#" * 90)

  file_name = directory.split('/')[-1] + ".xlsx"
  save_path = r"./"
  full_path = os.path.join(save_path, file_name)
  wb.save(full_path)

  print("Workbook successfully saved!")
  return full_path

In [ ]:
#@title Run the main evaluation function { display-mode: "form" }

if "lobachevsky" in directory.lower():
  print("Working On Lobachevsky Dataset")
  full_path=evaluate_ludb(record_names, directory)
else:
  print("Working on ", directory.split("/")[-1])
  full_path=evaluate_arr_nst_qt(record_names, directory)

Working on  mit-bih-arrhythmia-database-1.0.0
Working on: 100
Working on: 101
Working on: 102
Working on: 103
Working on: 104
Working on: 105
Working on: 106
Working on: 107
Working on: 108
Working on: 109
Working on: 111
Working on: 112
Working on: 113
Working on: 114
Working on: 115
Working on: 116
Working on: 117
Working on: 118
Working on: 119
Working on: 121
Working on: 122
Working on: 123
Working on: 124
Working on: 200
Working on: 201
Working on: 202
Working on: 203
Working on: 205
Working on: 207
Working on: 208
Working on: 209
Working on: 210
Working on: 212
Working on: 213
Working on: 214
Working on: 215
Working on: 217
Working on: 219
Working on: 220
Working on: 221
Working on: 222
Working on: 223
Working on: 228
Working on: 230
Working on: 231
Working on: 232
Working on: 233
Working on: 234

##########################################################################################
######################### GENERATING CONSOLE PERFORMANCE REPORT ##########################
###

In [ ]:
#@title **** Run this cell for statistical testing ****


###################################----statistical test---- #####################


# ==========================================
# 1. SELECT AND LOAD FILE VIA GUI
# ==========================================
# Hide the main tkinter root window



file_path = full_path

if not file_path:
    print("❌ No file selected. Exiting script.")
    exit()

print(f"Loading: {os.path.basename(file_path)}")
all_sheets = pd.read_excel(file_path, sheet_name=None)

# Extract the input file name without its extension (e.g., "LUDB")
base_filename = os.path.splitext(os.path.basename(file_path))[0]

# Set up a single unified output file destination using the filename
output_dir = os.path.dirname(file_path)
output_name = f"Nemenyi_Stat_{base_filename}.xlsx"
full_output_path = os.path.join(output_dir, output_name)

# Open the unified Excel writer context
with pd.ExcelWriter(full_output_path, engine='openpyxl') as writer:

    for sheet_name, df in all_sheets.items():
        print(f"\nReading Sheet: {sheet_name} .......")
        df.columns = df.columns.str.strip()

        # 1. Clean up potential spacing issues in data text
        if "Record" not in df.columns or "Method" not in df.columns:
            print(f"⚠️ Skipping sheet '{sheet_name}' because it lacks required 'Record' or 'Method' columns.")
            continue

        df["Record"] = df["Record"].astype(str).str.strip()
        df["Method"] = df["Method"].astype(str).str.strip()

        metric = "F1-Score"
        if metric not in df.columns:
            print(f"⚠️ Skipping sheet '{sheet_name}' because it lacks '{metric}' column.")
            continue

        df[metric] = pd.to_numeric(df[metric], errors='coerce')

        # ======================================================================
        # FIX: Create a temporary ID so rows with the same Record name don't merge
        # ======================================================================
        df['Row_ID'] = df['Record'] + "_" + df.groupby(['Record', 'Method']).cumcount().astype(str)

        # Pivot using the new unique Row_ID as the index instead of just "Record"
        pivot_df = df.pivot_table(index="Row_ID", columns="Method", values=metric, aggfunc='first')

        # Clean up any complete NaN rows if they exist
        pivot_df = pivot_df.dropna(how='all')

        print(f"Matrix generated for {metric} ({pivot_df.shape[0]} rows kept) ......")

        # ======================================================================
        # 2. RUN TESTS ON THE FULL DATA
        # ======================================================================
        # Ensure no NaN values interfere with Friedman Test calculation
        pivot_df = pivot_df.dropna()

        if pivot_df.shape[1] < 2:
            print(f"⚠️ Skipping sheet '{sheet_name}' - not enough valid methods left after dropping NaNs.")
            continue

        algo_data = [pivot_df[col].values for col in pivot_df.columns]

        stat, p_val = friedmanchisquare(*algo_data)
        print("\n===== FRIEDMAN TEST =====")
        print(f"Statistic: {stat:.4f}")
        print(f"p-value: {p_val:.6e}")

        if p_val < 0.05:
            print("\nSignificant difference among algorithms:True")
            print("So, running Nemenyi post-hoc test...")

            nemenyi_matrix = sp.posthoc_nemenyi_friedman(pivot_df.values)
            nemenyi_matrix.columns = pivot_df.columns
            nemenyi_matrix.index = pivot_df.columns

            print("\n===== NEMENYI P-VALUE MATRIX =====")
            print(nemenyi_matrix.round(5))

            # Create a clean sheet name (Excel sheets can have maximum 31 characters)
            clean_sheet_name = sheet_name.replace(' ', '_')[:31]

            # Save matrix directly into its corresponding sheet in the unified file
            nemenyi_matrix.to_excel(writer, sheet_name=clean_sheet_name)
            print(f" Added matrix to sheet: '{clean_sheet_name}'")
        else:
            print(f"\nSignificant difference among algorithms:False")
            print(f"{sheet_name}. Skipping Nemenyi sheet generation.")

print(f"\n Successfully saved all significance matrices to: {full_output_path}")

Loading: mit-bih-arrhythmia-database-1.0.0.xlsx

Reading Sheet: Clean_50ms .......
Matrix generated for F1-Score (48 rows kept) ......

===== FRIEDMAN TEST =====
Statistic: 98.4307
p-value: 1.131678e-19

Significant difference among algorithms:True
So, running Nemenyi post-hoc test...

===== NEMENYI P-VALUE MATRIX =====
Method       Christov  HTompkins  Neurokit  PanTompkins  Proposed  Wavelet
Method                                                                    
Christov      1.00000    1.00000   0.00186      0.00001   0.02968  0.89572
HTompkins     1.00000    1.00000   0.00231      0.00001   0.03497  0.91506
Neurokit      0.00186    0.00231   1.00000      0.00000   0.96913  0.06493
PanTompkins   0.00001    0.00001   0.00000      1.00000   0.00000  0.00000
Proposed      0.02968    0.03497   0.96913      0.00000   1.00000  0.36324
Wavelet       0.89572    0.91506   0.06493      0.00000   0.36324  1.00000
 Added matrix to sheet: 'Clean_50ms'

Reading Sheet: Clean_100ms .......
Matri